# Reject Inference under Class Imbalance with Label Spreading
Code for the master's thesis *Machine Learning for Credit Risk Modelling: A Framework Addressing Reject Inference and Class Imbalance* (NOVA IMS, 2026).

Place the LendingClub files in `data/` before running (see the README). Results are written to `results/`.

#### Import basic libraries

In [ ]:
## import basic libraries

import pandas as pd   
import matplotlib.pyplot as plt  
import numpy as np   
import seaborn as sns   
import itertools

import gc
import numpy as np
import pandas as pd
import scipy.sparse as sp

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.cluster import MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.semi_supervised import LabelSpreading
from sklearn.base import clone

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE, KMeansSMOTE

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

from scipy.stats import wilcoxon

import warnings
warnings.filterwarnings('ignore')

# Folders for input data and results
import os
os.makedirs('data', exist_ok=True)
for d in ['results/main', 'results/gamma_sweep', 'results/sensitivity', 'results/baseline', 'results/diagnostics']:
    os.makedirs(d, exist_ok=True)


#### Import the data

> **Rejected** data

In [ ]:
# read the data
df_rejected = pd.read_csv('data/rejected_2007_to_2018Q4.csv')

In [ ]:
df_rejected.info()

In [ ]:
# display of the first and last rows of the DataFrame
display(df_rejected.head(),df_rejected.tail())

> **Accepted** data

In [ ]:
# read the data
df_accepted = pd.read_csv('data/accepted_2007_to_2018Q4.csv')

In [ ]:
df_accepted.info()

In [ ]:
# display of the first and last rows of the DataFrame
display(df_accepted.head(),df_accepted.tail())

**Treatment for the Target Variable**

In [ ]:
df_accepted['loan_status'].value_counts()

In [ ]:
default_categories = [
    'Charged Off',
    'Default',
    'Does not meet the credit policy. Status:Charged Off',
    'Late (31-120 days)',
    'Late (16-30 days)'
]

df_accepted['default_flag'] = df_accepted['loan_status'].isin(default_categories).astype(int)

In [ ]:
df_accepted['default_flag'].value_counts()

In [ ]:
df_accepted['default_flag'].isnull().sum()

### Step 1 — Feature harmonization

In [ ]:
rename_mapping = {
    'Amount Requested': 'loan_amnt',
    'Debt-To-Income Ratio': 'dti',
    'Employment Length': 'emp_length',
    'State': 'addr_state'
}
df_rejected = df_rejected.rename(columns=rename_mapping)

shared_features = ['loan_amnt', 'dti', 'emp_length', 'addr_state']

df_rejected = df_rejected[shared_features]

In [ ]:
df_accepted = df_accepted[shared_features + ['default_flag']]

In [ ]:
print(df_rejected.columns)
print(df_accepted.columns)

In [ ]:
# Remove % sign and convert to float
df_rejected['dti'] = (
    df_rejected['dti']
    .str.replace('%', '', regex=False)
    .astype(float)
)

> ### Quality Investigation

#### Check for Duplicate Values

In [ ]:
n_duplicates = df_rejected.duplicated().sum()
print(f"You seem to have {n_duplicates} duplicates in your database.")

In [ ]:
# Check the number of duplicates  

n_duplicates2 = df_accepted.duplicated().sum()
print(f"You seem to have {n_duplicates2} duplicates in your database.")

#### Check for Missing Values

In [ ]:
# For each column, sum of missing values
print(df_rejected.isnull().sum())

In [ ]:
# For each column, sum of missing values
print(df_accepted.isnull().sum())

In [ ]:
df_accepted[df_accepted["loan_amnt"].isnull()]

In [ ]:
df_accepted=df_accepted.dropna(subset=["loan_amnt"])

In [ ]:
# For each column, sum of missing values
print(df_accepted.isnull().sum())

## Step 2 — Deterministic cleaning

- replace sentinel values (dti = -1 → NaN)

- standardize emp_length

- parse datatypes

In [ ]:
df_rejected[df_rejected["dti"]==-1]

In [ ]:
df_accepted[df_accepted["dti"]==-1]

In [ ]:
df_accepted['dti'] = df_accepted['dti'].replace(-1, np.nan)
df_rejected['dti'] = df_rejected['dti'].replace(-1, np.nan)

In [ ]:
df_accepted[df_accepted["dti"]==-1]

In [ ]:
emp_length_mapping = {
    '< 1 year': 0,
    '1 year': 1,
    '2 years': 2,
    '3 years': 3,
    '4 years': 4,
    '5 years': 5,
    '6 years': 6,
    '7 years': 7,
    '8 years': 8,
    '9 years': 9,
    '10+ years': 10,
}

df_accepted['emp_length'] = df_accepted['emp_length'].map(emp_length_mapping)
df_rejected['emp_length'] = df_rejected['emp_length'].map(emp_length_mapping)

In [ ]:
df_rejected["emp_length"].value_counts()

In [ ]:
df_rejected.info()

In [ ]:
df_accepted.info()

## Proposed Framework

### Pipeline

In [ ]:
# ============================================================
# Global feature definitions
# ============================================================

NUM_COLS = ["loan_amnt", "dti", "emp_length"]
CAT_COLS = ["addr_state"]
FEATURES = NUM_COLS + CAT_COLS


# ============================================================
# Helper: OneHotEncoder compatibility
# ============================================================

def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


# ============================================================
# Helper: fit preprocessing objects
# ============================================================

def fit_preprocessing_objects(X_acc_ri_raw):
    """
    Fits preprocessing only on the accepted RI sample.

    Important:
    - Numeric preprocessing is used for oversampling and final graph representation.
    - Categorical preprocessing is fitted here, but one-hot encoding is only applied
      after categorical-safe oversampling.
    """

    num_imputer = SimpleImputer(strategy="median")
    num_scaler = StandardScaler()

    cat_imputer = SimpleImputer(strategy="most_frequent")
    onehot = make_onehot_encoder()

    # Fit numeric preprocessing
    X_num_imp = num_imputer.fit_transform(X_acc_ri_raw[NUM_COLS])
    X_num_scaled = num_scaler.fit_transform(X_num_imp)

    # Fit categorical preprocessing
    X_cat_imp = cat_imputer.fit_transform(X_acc_ri_raw[CAT_COLS])
    onehot.fit(X_cat_imp)

    return {
        "num_imputer": num_imputer,
        "num_scaler": num_scaler,
        "cat_imputer": cat_imputer,
        "onehot": onehot,
    }


# ============================================================
# Helper: transform numerical part only
# ============================================================

def transform_numeric(X_raw, pp):
    """
    Impute and scale numerical/ordinal features only.
    Used for SMOTE/KMeans-SMOTE and nearest-neighbor state assignment.
    """
    X_num_imp = pp["num_imputer"].transform(X_raw[NUM_COLS])
    X_num_scaled = pp["num_scaler"].transform(X_num_imp)
    return X_num_scaled


# ============================================================
# Helper: transform final full feature representation
# ============================================================

def transform_final(X_raw, pp):
    """
    Final representation used for:
    - rejected sampling/clustering
    - Label Spreading
    - supervised classifiers

    Numeric features are scaled.
    addr_state is one-hot encoded.
    """
    X_num_scaled = transform_numeric(X_raw, pp)

    X_cat_imp = pp["cat_imputer"].transform(X_raw[CAT_COLS])
    X_cat_oh = pp["onehot"].transform(X_cat_imp)

    return sp.hstack(
        [sp.csr_matrix(X_num_scaled), X_cat_oh],
        format="csr"
    )


# ============================================================
# Helper: get imputed state labels
# ============================================================

def get_imputed_states(X_raw, pp):
    """
    Returns imputed addr_state values as a 1D array.
    """
    X_cat_imp = pp["cat_imputer"].transform(X_raw[CAT_COLS])
    return X_cat_imp.ravel()


# ============================================================
# Helper: assign addr_state to synthetic samples
# ============================================================

def assign_states_to_synthetic(
    X_num_os,
    y_os,
    X_num_original,
    y_original,
    state_original
):
    """
    SMOTE/KMeans-SMOTE are applied only to numeric/ordinal variables.
    This function assigns a valid addr_state to synthetic observations.

    Original observations keep their original addr_state.
    Synthetic observations receive addr_state from the nearest original
    minority-class borrower in scaled numerical space.

    This avoids invalid fractional one-hot states 
    """

    y_original_arr = np.asarray(y_original, dtype=int)
    y_os_arr = np.asarray(y_os, dtype=int)

    n_original = X_num_original.shape[0]
    n_total = X_num_os.shape[0]
    n_synth = n_total - n_original

    if n_synth < 0:
        raise ValueError("Oversampled data has fewer rows than original data.")

    # Check that imblearn kept original samples first
    if not np.array_equal(y_os_arr[:n_original], y_original_arr):
        raise RuntimeError(
            "Unexpected sample ordering after oversampling. "
            "The code assumes original samples appear first in the resampled matrix."
        )

    # Original rows keep original states
    state_os = list(np.asarray(state_original))

    # No synthetic rows generated
    if n_synth == 0:
        return np.asarray(state_os)

    # Synthetic numerical rows
    X_synth = X_num_os[n_original:]

    # Nearest original minority-class samples
    minority_mask = y_original_arr == 1

    if minority_mask.sum() == 0:
        raise ValueError("No minority-class samples found for state assignment.")

    X_minority = X_num_original[minority_mask]
    state_minority = np.asarray(state_original)[minority_mask]

    nn = NearestNeighbors(n_neighbors=1)
    nn.fit(X_minority)

    nearest_idx = nn.kneighbors(X_synth, return_distance=False).ravel()
    synth_states = state_minority[nearest_idx]

    state_os.extend(synth_states.tolist())

    return np.asarray(state_os)


# ============================================================
# Helper: build accepted oversampled final matrix
# ============================================================

def build_accepted_oversampled_matrix(
    X_num_os,
    y_os,
    X_num_original,
    y_original,
    state_original,
    pp
):
    """
    Builds final accepted oversampled feature matrix:
    - Numeric part comes from SMOTE/KMeans-SMOTE output.
    - Categorical state is assigned validly and then one-hot encoded.
    """

    state_os = assign_states_to_synthetic(
        X_num_os=X_num_os,
        y_os=y_os,
        X_num_original=X_num_original,
        y_original=y_original,
        state_original=state_original
    )

    X_state_oh = pp["onehot"].transform(state_os.reshape(-1, 1))

    X_final = sp.hstack(
        [sp.csr_matrix(X_num_os), X_state_oh],
        format="csr"
    )

    return X_final


# ============================================================
# Helper: rejected sampling
# ============================================================

def rejected_random_sample(df_rej, S, seed):
    return df_rej.sample(n=S, random_state=seed)


def rejected_cluster_sample(df_rej_pp, df_rej_raw, S, seed, n_clusters=100):

    sub_n = min(200_000, df_rej_pp.shape[0])
    rng = np.random.RandomState(seed)

    idx = rng.choice(
        df_rej_pp.shape[0],
        size=sub_n,
        replace=False
    )

    kmeans = MiniBatchKMeans(
        n_clusters=n_clusters,
        random_state=seed,
        batch_size=10_000,
        n_init=5
    )

    kmeans.fit(df_rej_pp[idx])

    # Assign all rejected applicants to clusters
    cl = kmeans.predict(df_rej_pp)

    # Proportional sampling by cluster
    counts = np.bincount(cl, minlength=n_clusters)
    props = counts / counts.sum()

    alloc = np.floor(props * S).astype(int)

    remainder = S - alloc.sum()
    if remainder > 0:
        frac = props * S - np.floor(props * S)
        extra_idx = np.argsort(frac)[::-1][:remainder]
        alloc[extra_idx] += 1

    sampled_indices = []

    for c in range(n_clusters):
        n_c = alloc[c]

        if n_c <= 0:
            continue

        members = np.where(cl == c)[0]

        if len(members) <= n_c:
            sampled_indices.extend(members.tolist())
        else:
            sampled_indices.extend(
                rng.choice(members, size=n_c, replace=False).tolist()
            )

    sampled_indices = np.asarray(sampled_indices)

    return df_rej_raw.iloc[sampled_indices]


# ============================================================
# Helper: vertical stack
# ============================================================

def vstack_any(A, B):
    """
    Vertically stack A and B whether they are sparse or dense.
    """
    if sp.issparse(A) or sp.issparse(B):
        A = sp.csr_matrix(A) if not sp.issparse(A) else A.tocsr()
        B = sp.csr_matrix(B) if not sp.issparse(B) else B.tocsr()
        return sp.vstack([A, B], format="csr")
    else:
        return np.vstack([A, B])


# ============================================================
# Helper: train and evaluate classifier
# ============================================================

def eval_model(clf, X_train, y_train, X_test, y_test):

    clf.fit(X_train, y_train)

    p = clf.predict_proba(X_test)[:, 1]

    return {
        "roc_auc": roc_auc_score(y_test, p),
        "pr_auc": average_precision_score(y_test, p),
        "brier": brier_score_loss(y_test, p),
    }


# ============================================================
# Main pipeline
# ============================================================

def run_one_seed(
    seed,
    df_accepted,
    df_rejected,
    N_L=10_000,
    S=3_000,
    gamma=2.0
):

    # ------------------------------------------------------------
    # 1) Accepted split: 80/20 stratified
    # ------------------------------------------------------------

    X = df_accepted[FEATURES]
    y = df_accepted["default_flag"].astype(int)

    X_prelim_full, X_test, y_prelim_full, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        stratify=y,
        random_state=seed
    )

    # ------------------------------------------------------------
    # 2) Subsample accepted preliminary training set to N_L
    # ------------------------------------------------------------

    X_prelim_RI, _, y_prelim_RI, _ = train_test_split(
        X_prelim_full,
        y_prelim_full,
        train_size=N_L,
        stratify=y_prelim_full,
        random_state=seed
    )

    y_prelim_RI_arr = np.asarray(y_prelim_RI, dtype=int)

    # ------------------------------------------------------------
    # 3) Fit preprocessing on accepted RI sample only
    # ------------------------------------------------------------

    pp = fit_preprocessing_objects(X_prelim_RI)

    X_num_RI = transform_numeric(X_prelim_RI, pp)
    state_RI = get_imputed_states(X_prelim_RI, pp)

    X_test_pp = transform_final(X_test, pp)

    # ------------------------------------------------------------
    # 4) Transform full rejected pool for rejected sampling/clustering
    # ------------------------------------------------------------

    X_rej_raw = df_rejected[FEATURES]
    X_rej_pp = transform_final(X_rej_raw, pp)

    # ------------------------------------------------------------
    # 5) Rejected sampling
    # ------------------------------------------------------------

    rej_rand_raw = rejected_random_sample(
        X_rej_raw,
        S=S,
        seed=seed
    )

    rej_clust_raw = rejected_cluster_sample(
        df_rej_pp=X_rej_pp,
        df_rej_raw=X_rej_raw,
        S=S,
        seed=seed,
        n_clusters=100
    )

    X_rej_rand_pp = transform_final(rej_rand_raw, pp)
    X_rej_cluster_pp = transform_final(rej_clust_raw, pp)

    # ------------------------------------------------------------
    # 6) Oversampling on accepted RI sample only
    #    IMPORTANT: only numeric/ordinal variables are oversampled.
    # ------------------------------------------------------------

    smote = SMOTE(
        sampling_strategy=0.5,
        k_neighbors=5,
        random_state=seed
    )

    X_num_std_os, y_acc_std_os = smote.fit_resample(
        X_num_RI,
        y_prelim_RI_arr
    )

    kmeans_est = MiniBatchKMeans(
        n_clusters=30,
        random_state=seed,
        batch_size=10_000,
        n_init=5
    )

    kmeans_smote = KMeansSMOTE(
        sampling_strategy=0.5,
        k_neighbors=5,
        kmeans_estimator=kmeans_est,
        cluster_balance_threshold=0.1,
        random_state=seed
    )

    X_num_nh_os, y_acc_nh_os = kmeans_smote.fit_resample(
        X_num_RI,
        y_prelim_RI_arr
    )

    # ------------------------------------------------------------
    # 7) Rebuild accepted oversampled matrices with valid addr_state
    # ------------------------------------------------------------

    X_acc_std_os = build_accepted_oversampled_matrix(
        X_num_os=X_num_std_os,
        y_os=y_acc_std_os,
        X_num_original=X_num_RI,
        y_original=y_prelim_RI_arr,
        state_original=state_RI,
        pp=pp
    )

    X_acc_nh_os = build_accepted_oversampled_matrix(
        X_num_os=X_num_nh_os,
        y_os=y_acc_nh_os,
        X_num_original=X_num_RI,
        y_original=y_prelim_RI_arr,
        state_original=state_RI,
        pp=pp
    )

    # ------------------------------------------------------------
    # 8) Build graph datasets: accepted labeled + rejected unlabeled
    # ------------------------------------------------------------

    y_unlab = -1 * np.ones(S, dtype=int)

    X_std_graph = vstack_any(X_acc_std_os, X_rej_rand_pp)
    y_std_graph = np.concatenate([
        np.asarray(y_acc_std_os, dtype=int),
        y_unlab
    ])

    X_nh_graph = vstack_any(X_acc_nh_os, X_rej_cluster_pp)
    y_nh_graph = np.concatenate([
        np.asarray(y_acc_nh_os, dtype=int),
        y_unlab
    ])

    # ------------------------------------------------------------
    # 9) Label Spreading
    # ------------------------------------------------------------

    ls_std = LabelSpreading(
        kernel="rbf",
        gamma=gamma,
        alpha=0.5,
        max_iter=100,
        tol=1e-3,
        n_jobs=-1
    )

    ls_std.fit(X_std_graph, y_std_graph)

    F_std = ls_std.label_distributions_
    p_rej_std = F_std[-S:, 1]
    y_rej_std = (p_rej_std >= 0.5).astype(int)

    eps = 1e-12
    rej_probs_std = F_std[-S:, :]

    conf_std = np.max(rej_probs_std, axis=1).mean()
    entropy_std = (
        -np.sum(rej_probs_std * np.log(rej_probs_std + eps), axis=1)
    ).mean()
    rej_prob_rate_std = p_rej_std.mean()
    rej_hard_rate_std = y_rej_std.mean()

    ls_nh = LabelSpreading(
        kernel="rbf",
        gamma=gamma,
        alpha=0.5,
        max_iter=100,
        tol=1e-3,
        n_jobs=-1
    )

    ls_nh.fit(X_nh_graph, y_nh_graph)

    F_nh = ls_nh.label_distributions_
    p_rej_nh = F_nh[-S:, 1]
    y_rej_nh = (p_rej_nh >= 0.5).astype(int)

    rej_probs_nh = F_nh[-S:, :]

    conf_nh = np.max(rej_probs_nh, axis=1).mean()
    entropy_nh = (
        -np.sum(rej_probs_nh * np.log(rej_probs_nh + eps), axis=1)
    ).mean()
    rej_prob_rate_nh = p_rej_nh.mean()
    rej_hard_rate_nh = y_rej_nh.mean()

    # ------------------------------------------------------------
    # 10) Final training sets: accepted oversampled + inferred rejected
    # ------------------------------------------------------------

    X_train_std_final = vstack_any(X_acc_std_os, X_rej_rand_pp)
    y_train_std_final = np.concatenate([
        np.asarray(y_acc_std_os, dtype=int),
        y_rej_std
    ])

    X_train_nh_final = vstack_any(X_acc_nh_os, X_rej_cluster_pp)
    y_train_nh_final = np.concatenate([
        np.asarray(y_acc_nh_os, dtype=int),
        y_rej_nh
    ])

    # Sanity checks
    assert (y_train_std_final == -1).sum() == 0, \
        "STD: found unlabeled (-1) labels in final set."

    assert (y_train_nh_final == -1).sum() == 0, \
        "NH: found unlabeled (-1) labels in final set."

    assert X_train_std_final.shape[0] == len(y_train_std_final), \
        "STD: X/y shape mismatch."

    assert X_train_nh_final.shape[0] == len(y_train_nh_final), \
        "NH: X/y shape mismatch."

    assert X_rej_rand_pp.shape[0] == S, \
        "STD: rejected sample size mismatch."

    assert X_rej_cluster_pp.shape[0] == S, \
        "NH: rejected sample size mismatch."

    print(f"Seed {seed}: Sanity checks passed")

    # ------------------------------------------------------------
    # 11) Models
    # ------------------------------------------------------------

    models = {
        "LR": LogisticRegression(
            solver="lbfgs",
            max_iter=1000,
            random_state=seed
        ),

        "RF": RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=50,
            n_jobs=-1,
            random_state=seed
        ),

        "GBDT": GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            random_state=seed
        ),

        "XGBoost": XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=seed,
            n_jobs=-1
        ),

        "MLP": MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            solver="adam",
            alpha=1e-4,
            batch_size=256,
            learning_rate_init=1e-3,
            max_iter=200,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=10,
            random_state=seed
        ),
    }

    # ------------------------------------------------------------
    # 12) Evaluation
    # ------------------------------------------------------------

    rows = []

    for name, clf in models.items():

        std_metrics = eval_model(
            clone(clf),
            X_train_std_final,
            y_train_std_final,
            X_test_pp,
            y_test
        )

        nh_metrics = eval_model(
            clone(clf),
            X_train_nh_final,
            y_train_nh_final,
            X_test_pp,
            y_test
        )

        rows.append({
            "seed": seed,
            "N_L": N_L,
            "S": S,
            "gamma": gamma,
            "model": name,
            "pipeline": "STD",
            **std_metrics,
            "rej_mean_conf": np.nan,
            "rej_mean_entropy": np.nan,
            "rej_default_rate_prob": np.nan,
            "rej_default_rate_hard": np.nan,
        })

        rows.append({
            "seed": seed,
            "N_L": N_L,
            "S": S,
            "gamma": gamma,
            "model": name,
            "pipeline": "NH",
            **nh_metrics,
            "rej_mean_conf": np.nan,
            "rej_mean_entropy": np.nan,
            "rej_default_rate_prob": np.nan,
            "rej_default_rate_hard": np.nan,
        })

    # ------------------------------------------------------------
    # 13) Stability rows
    # ------------------------------------------------------------

    rows.append({
        "seed": seed,
        "N_L": N_L,
        "S": S,
        "gamma": gamma,
        "model": "LABEL",
        "pipeline": "STD",
        "roc_auc": np.nan,
        "pr_auc": np.nan,
        "brier": np.nan,
        "rej_mean_conf": conf_std,
        "rej_mean_entropy": entropy_std,
        "rej_default_rate_prob": rej_prob_rate_std,
        "rej_default_rate_hard": rej_hard_rate_std,
    })

    rows.append({
        "seed": seed,
        "N_L": N_L,
        "S": S,
        "gamma": gamma,
        "model": "LABEL",
        "pipeline": "NH",
        "roc_auc": np.nan,
        "pr_auc": np.nan,
        "brier": np.nan,
        "rej_mean_conf": conf_nh,
        "rej_mean_entropy": entropy_nh,
        "rej_default_rate_prob": rej_prob_rate_nh,
        "rej_default_rate_hard": rej_hard_rate_nh,
    })

    out = pd.DataFrame(rows)

    # ------------------------------------------------------------
    # 14) Cleanup
    # ------------------------------------------------------------

    del ls_std, ls_nh
    del X_std_graph, y_std_graph, X_nh_graph, y_nh_graph
    del F_std, F_nh
    gc.collect()

    return out

### Main predictive evaluation

In [ ]:
best_gamma = 2 
out_path = "results/main/final_20seeds_v2_bestgamma.csv"
first = True

for s in range(20):
    df_run = run_one_seed(s, df_accepted, df_rejected, N_L=10_000, S=3_000, gamma=best_gamma)
    df_run.to_csv(out_path, mode="w" if first else "a", header=first, index=False)
    first = False
    del df_run
    gc.collect()

final_runs = pd.read_csv(out_path)

In [ ]:
perf_runs = final_runs[final_runs["model"] != "LABEL"]

perf_summary = (perf_runs
    .groupby(["gamma","model","pipeline"])[["roc_auc","pr_auc","brier"]]
    .agg(["mean","std"])
    .round(4)
)
perf_summary

### Accepted-size sensitivity

This experiment evaluates whether the results are **sensitive** to the number of **labeled** accepted applicants used for graph construction and downstream training.

Fixed:
- gamma = 2
- S = 3,000

Varied:
- N_L = 10k, 20k, 30k, 50k

In [ ]:
best_gamma = 2
N_L_value = 10000  ### Change Here [10k, 20k, 30k, 50k]
S_value = 3000     ### Keep Fixed
out_path = f"results/sensitivity/perf_gamma2_NL{N_L_value}_S{S_value}.csv"
first = True

for s in range(20):
    df_run = run_one_seed(
        s, 
        df_accepted, 
        df_rejected, 
        N_L=N_L_value,  
        S=S_value, 
        gamma=best_gamma
    )
    df_run.to_csv(out_path, mode="w" if first else "a", header=first, index=False)
    first = False
    del df_run
    gc.collect()

final_runs = pd.read_csv(out_path)

In [ ]:
perf_runs = final_runs[final_runs["model"] != "LABEL"]

perf_summary = (perf_runs
    .groupby(["gamma","model","pipeline"])[["roc_auc","pr_auc","brier"]]
    .agg(["mean","std"])
    .round(4)
)
perf_summary

### Rejected-size sensitivity

This experiment evaluates whether the results are **sensitive** to the number of sampled **rejected** applicants included in the Label Spreading graph.

Fixed:
- gamma = 2
- N_L = 10,000

Varied:
- S = 1k, 3k, 5k

In [ ]:
best_gamma = 2
N_L_value = 10000  ### Keep Fixed
S_value = 5000     ### Change Here [1k, 3k, 5k]
out_path = f"results/sensitivity/perf_gamma2_NL{N_L_value}_S{S_value}.csv"
first = True

for s in range(20):
    df_run = run_one_seed(
        s, 
        df_accepted, 
        df_rejected, 
        N_L=N_L_value,  
        S=S_value,     
        gamma=best_gamma
    )
    df_run.to_csv(out_path, mode="w" if first else "a", header=first, index=False)
    first = False
    del df_run
    gc.collect()

final_runs = pd.read_csv(out_path)

In [ ]:
perf_runs = final_runs[final_runs["model"] != "LABEL"]

perf_summary = (perf_runs
    .groupby(["gamma","model","pipeline"])[["roc_auc","pr_auc","brier"]]
    .agg(["mean","std"])
    .round(4)
)
perf_summary

### Label Propagation behavior analysis

This was the method used to choose the gamma to use on the performance evaluation.

Default:
- gamma = [0.5, 1, 2, 5, 10]
- 10 seeds
- N_L = 10,000
- S = 3,000

In [ ]:
gammas = [0.5, 1, 2, 5, 10]
out_path = "results/gamma_sweep/gamma_sweep2_results.csv"
first = True

for g in gammas:
    for s in range(10):
        df_run = run_one_seed(
            s, 
            df_accepted, 
            df_rejected, 
            N_L=10_000,   ### Change Here If Needed
            S=3_000,      ### Change Here If Needed
            gamma=g
        )
        df_run.to_csv(out_path, mode="w" if first else "a", header=first, index=False)
        first = False
        del df_run
        gc.collect()

all_runs = pd.read_csv(out_path)

In [ ]:
label_runs = all_runs[all_runs["model"] == "LABEL"]

stability_summary = (label_runs
    .groupby(["gamma","pipeline"])[["rej_mean_conf","rej_mean_entropy","rej_default_rate_prob","rej_default_rate_hard"]]
    .agg(["mean","std"])
    .round(4)
)
stability_summary

### Statistical Test (Wilcoxon)

“Are the observed differences likely due to the pipeline design rather than random seed variation?”

“Is the performance difference between NH and STD systematic across seeds, or could it be due to randomness?”

Wilcoxon signed-rank test (paired, non-parametric)

In [ ]:
def wilcoxon_by_model(df, metric="roc_auc", alternative="two-sided"):
    results = []

    for model in df["model"].unique():
        df_m = df[df["model"] == model]

        std = (
            df_m[df_m["pipeline"] == "STD"]
            .sort_values("seed")[metric]
            .values
        )
        nh = (
            df_m[df_m["pipeline"] == "NH"]
            .sort_values("seed")[metric]
            .values
        )

        # Sanity check: paired samples
        assert len(std) == len(nh), "Unpaired samples detected"

        stat, p = wilcoxon(nh, std, alternative=alternative)

        results.append({
            "model": model,
            "metric": metric,
            "wilcoxon_stat": stat,
            "p_value": p,
            "nh_mean": nh.mean(),
            "std_mean": std.mean(),
            "mean_diff": nh.mean() - std.mean()
        })

    return pd.DataFrame(results)


In [ ]:
# Always test the main experiment (the sensitivity cells above overwrite `final_runs`)
final_runs = pd.read_csv("results/main/final_20seeds_v2_bestgamma.csv")

wix_runs = final_runs[final_runs["model"] != "LABEL"]

wilcoxon_roc = wilcoxon_by_model(wix_runs, metric="roc_auc")

wilcoxon_roc

In [ ]:
def wilcoxon_by_model_2(df, metric="pr_auc", alternative="two-sided"):
    results = []

    for model in df["model"].unique():
        df_m = df[df["model"] == model]

        std = (
            df_m[df_m["pipeline"] == "STD"]
            .sort_values("seed")[metric]
            .values
        )
        nh = (
            df_m[df_m["pipeline"] == "NH"]
            .sort_values("seed")[metric]
            .values
        )

        # Sanity check: paired samples
        assert len(std) == len(nh), "Unpaired samples detected"

        stat, p = wilcoxon(nh, std, alternative=alternative)

        results.append({
            "model": model,
            "metric": metric,
            "wilcoxon_stat": stat,
            "p_value": p,
            "nh_mean": nh.mean(),
            "std_mean": std.mean(),
            "mean_diff": nh.mean() - std.mean()
        })

    return pd.DataFrame(results)

In [ ]:
# Always test the main experiment (the sensitivity cells above overwrite `final_runs`)
final_runs = pd.read_csv("results/main/final_20seeds_v2_bestgamma.csv")

wix_runs = final_runs[final_runs["model"] != "LABEL"]

wilcoxon_roc = wilcoxon_by_model_2(wix_runs, metric="pr_auc")
wilcoxon_roc

### Holm correction for multiple testing
Adjusts the 10 Wilcoxon p-values (Section 4.5 of the thesis).

In [ ]:
# Holm correction across the 10 Wilcoxon tests (5 classifiers x ROC-AUC and PR-AUC)
runs_main = final_runs[final_runs["model"] != "LABEL"]
all_tests = pd.concat([wilcoxon_by_model(runs_main, metric="roc_auc"),
                       wilcoxon_by_model_2(runs_main, metric="pr_auc")], ignore_index=True)

pcol = "p_value"
order = all_tests[pcol].sort_values().index
m = len(all_tests); running_max = 0.0; holm = {}
for rank, idx in enumerate(order):
    running_max = max(running_max, min(1.0, (m - rank) * all_tests.loc[idx, pcol]))
    holm[idx] = running_max
all_tests["p_holm"] = pd.Series(holm)
all_tests["significant_holm"] = all_tests["p_holm"] < 0.05
all_tests


### Accepted-only baseline
Each pipeline compared with an identical twin trained without rejected applicants (Section 4.7 of the thesis).

In [ ]:
# ============================================================
# ACCEPTED-ONLY BASELINE
# Paste as a NEW cell after cell 42 (the pipeline cell) and run it.
# It reuses your helpers unchanged:
#   NUM_COLS, CAT_COLS, FEATURES, fit_preprocessing_objects,
#   transform_numeric, transform_final, get_imputed_states,
#   build_accepted_oversampled_matrix, eval_model
#
# Steps 1-3 and 6-7 are identical to run_one_seed (same seeds, same split,
# same N_L sample, same SMOTE / KMeans-SMOTE), so each baseline is the
# exact twin of a pipeline with the rejected applicants removed.
# No rejected data, no Label Spreading.
#
# Three baselines per seed and classifier:
#   ACC_SMOTE   = STD pipeline without rejected applicants
#   ACC_KMSMOTE = NH pipeline without rejected applicants
#   ACC_RAW     = accepted sample only, no oversampling, no rejected
# ============================================================

def run_one_seed_baseline(seed, df_accepted, N_L=10_000):

    # 1) Accepted split: 80/20 stratified  (identical to run_one_seed)
    X = df_accepted[FEATURES]
    y = df_accepted["default_flag"].astype(int)

    X_prelim_full, X_test, y_prelim_full, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed
    )

    # 2) Subsample accepted preliminary training set to N_L  (identical)
    X_prelim_RI, _, y_prelim_RI, _ = train_test_split(
        X_prelim_full, y_prelim_full,
        train_size=N_L, stratify=y_prelim_full, random_state=seed
    )
    y_prelim_RI_arr = np.asarray(y_prelim_RI, dtype=int)

    # 3) Preprocessing fitted on the accepted sample only  (identical)
    pp = fit_preprocessing_objects(X_prelim_RI)
    X_num_RI = transform_numeric(X_prelim_RI, pp)
    state_RI = get_imputed_states(X_prelim_RI, pp)
    X_test_pp = transform_final(X_test, pp)
    X_acc_raw = transform_final(X_prelim_RI, pp)          # for ACC_RAW

    # 6) Oversampling on accepted sample only  (identical settings)
    smote = SMOTE(sampling_strategy=0.5, k_neighbors=5, random_state=seed)
    X_num_std_os, y_acc_std_os = smote.fit_resample(X_num_RI, y_prelim_RI_arr)

    kmeans_est = MiniBatchKMeans(
        n_clusters=30, random_state=seed, batch_size=10_000, n_init=5
    )
    kmeans_smote = KMeansSMOTE(
        sampling_strategy=0.5, k_neighbors=5, kmeans_estimator=kmeans_est,
        cluster_balance_threshold=0.1, random_state=seed
    )
    X_num_nh_os, y_acc_nh_os = kmeans_smote.fit_resample(X_num_RI, y_prelim_RI_arr)

    # 7) Rebuild oversampled matrices with valid addr_state  (identical)
    X_acc_std_os = build_accepted_oversampled_matrix(
        X_num_std_os, y_acc_std_os, X_num_RI, y_prelim_RI_arr, state_RI, pp
    )
    X_acc_nh_os = build_accepted_oversampled_matrix(
        X_num_nh_os, y_acc_nh_os, X_num_RI, y_prelim_RI_arr, state_RI, pp
    )

    # Steps 4, 5, 8, 9 and 10 of run_one_seed are skipped:
    # the training sets are the oversampled accepted data only.
    train_sets = {
        "ACC_SMOTE":   (X_acc_std_os, np.asarray(y_acc_std_os, dtype=int)),
        "ACC_KMSMOTE": (X_acc_nh_os,  np.asarray(y_acc_nh_os,  dtype=int)),
        "ACC_RAW":     (X_acc_raw,    y_prelim_RI_arr),
    }

    # 11) Models  (copied exactly from run_one_seed - keep them identical)
    models = {
        "LR": LogisticRegression(solver="lbfgs", max_iter=1000, random_state=seed),
        "RF": RandomForestClassifier(
            n_estimators=300, max_depth=None, min_samples_leaf=50,
            n_jobs=-1, random_state=seed
        ),
        "GBDT": GradientBoostingClassifier(
            n_estimators=200, learning_rate=0.05, max_depth=3, random_state=seed
        ),
        "XGBoost": XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
            colsample_bytree=0.8, eval_metric="logloss", random_state=seed, n_jobs=-1
        ),
        "MLP": MLPClassifier(
            hidden_layer_sizes=(64, 32), activation="relu", solver="adam",
            alpha=1e-4, batch_size=256, learning_rate_init=1e-3, max_iter=200,
            early_stopping=True, validation_fraction=0.1, n_iter_no_change=10,
            random_state=seed
        ),
    }

    # 12) Evaluation on the same 20% accepted test set
    rows = []
    for name, clf in models.items():
        for baseline, (X_tr, y_tr) in train_sets.items():
            metrics = eval_model(clone(clf), X_tr, y_tr, X_test_pp, y_test)
            rows.append({
                "seed": seed, "N_L": N_L, "S": 0, "gamma": np.nan,
                "model": name, "pipeline": baseline, **metrics,
                "n_train": X_tr.shape[0], "train_default_rate": y_tr.mean(),
            })

    print(f"Seed {seed}: baseline done")
    return pd.DataFrame(rows)


# ============================================================
# Driver - NEW cell, same 20 seeds as the main experiment
# ============================================================

out_path_base = "results/baseline/baseline_accepted_only_20seeds.csv"
first = True

for s in range(20):
    df_run = run_one_seed_baseline(s, df_accepted, N_L=10_000)
    df_run.to_csv(out_path_base, mode="w" if first else "a", header=first, index=False)
    first = False
    del df_run
    gc.collect()

baseline_runs = pd.read_csv(out_path_base)
baseline_runs.groupby(["model", "pipeline"])[["roc_auc", "pr_auc", "brier"]].agg(["mean", "std"]).round(4)


### Isolated-applicant diagnostic
Counts rejected applicants that receive no label in the Label Spreading graph because they are too far from every other node.

In [ ]:
# ============================================================
# DIAGNOSTIC: rejected applicants isolated in the Label Spreading graph
#
# Paste as a NEW cell after cell 42 (the pipeline cell), once
# df_accepted and df_rejected exist. It does NOT change run_one_seed:
# it temporarily replaces two names that run_one_seed uses,
#   - LabelSpreading -> same class, but records the rejected nodes' labels
#   - eval_model     -> skipped (no classifier training, so it runs faster)
# and restores both at the end, even if something fails.
#
# A rejected applicant is "isolated" when its RBF similarity to every
# other node underflows to zero. scikit-learn then gives it the label
# distribution [0, 0]: confidence 0, default probability 0, hard label 0.
# ============================================================

import gc
import numpy as np
import pandas as pd
from sklearn.semi_supervised import LabelSpreading as _LabelSpreadingOriginal

_ls_records = []
_current_seed = None


class LabelSpreadingDiag(_LabelSpreadingOriginal):
    """Identical to LabelSpreading; also records what happens to unlabeled nodes."""

    def fit(self, X, y):
        super().fit(X, y)
        y = np.asarray(y)
        unl = y == -1
        F = self.label_distributions_[unl]          # rejected applicants only
        isolated = F.sum(axis=1) == 0
        ok = ~isolated

        # First 3 columns of the graph matrix = standardized loan_amnt, dti, emp_length
        X_unl = X[unl]
        X_num = X_unl[:, :3].toarray() if hasattr(X_unl, "toarray") else np.asarray(X_unl[:, :3])
        z_iso = np.abs(X_num[isolated])

        # run_one_seed fits STD first, then NH
        n_calls_this_seed = sum(r["seed"] == _current_seed for r in _ls_records)
        _ls_records.append({
            "seed": _current_seed,
            "pipeline": "STD" if n_calls_this_seed == 0 else "NH",
            "n_rejected": int(unl.sum()),
            "n_isolated": int(isolated.sum()),
            "share_isolated": isolated.mean(),
            # metrics as reported in the thesis (all rejected applicants)
            "conf_all": F.max(axis=1).mean(),
            "hard_default_all": (F[:, 1] >= 0.5).mean(),
            # the same metrics without the isolated applicants
            "conf_non_isolated": F[ok].max(axis=1).mean() if ok.any() else np.nan,
            "hard_default_non_isolated": (F[ok, 1] >= 0.5).mean() if ok.any() else np.nan,
            # how extreme the isolated applicants are (in standard deviations)
            "isolated_max_z_loan_amnt": z_iso[:, 0].max() if isolated.any() else np.nan,
            "isolated_max_z_dti": z_iso[:, 1].max() if isolated.any() else np.nan,
            "isolated_min_largest_z": z_iso.max(axis=1).min() if isolated.any() else np.nan,
        })
        return self


def _eval_model_skip(clf, X_train, y_train, X_test, y_test):
    return {"roc_auc": np.nan, "pr_auc": np.nan, "brier": np.nan}


# ---------- run: same settings and seeds as the main experiment ----------
_eval_model_original = eval_model
LabelSpreading = LabelSpreadingDiag
eval_model = _eval_model_skip
try:
    for s in range(20):                      # use range(5) for a quicker first look
        _current_seed = s
        _ = run_one_seed(s, df_accepted, df_rejected, N_L=10_000, S=3_000, gamma=2.0)
        print(f"Seed {s} done")
        del _
        gc.collect()
finally:
    LabelSpreading = _LabelSpreadingOriginal
    eval_model = _eval_model_original

diag = pd.DataFrame(_ls_records)
diag.to_csv("results/diagnostics/isolated_rejected_diagnostic.csv", index=False)

# ---------- 1) Summary per pipeline ----------
summary_cols = ["n_isolated", "share_isolated", "conf_all", "conf_non_isolated",
                "hard_default_all", "hard_default_non_isolated"]
print(diag.groupby("pipeline")[summary_cols].agg(["mean", "min", "max"]).round(4).T)

# ---------- 2) Sanity check: must reproduce the reported label metrics ----------
main = pd.read_csv("results/main/final_20seeds_v2_bestgamma.csv")
lab = main[main["model"] == "LABEL"][["seed", "pipeline", "rej_mean_conf", "rej_default_rate_hard"]]
check = diag.merge(lab, on=["seed", "pipeline"])
check["conf_matches"] = np.isclose(check["conf_all"], check["rej_mean_conf"], atol=1e-6)
check["hard_matches"] = np.isclose(check["hard_default_all"], check["rej_default_rate_hard"], atol=1e-6)
print("Reproduces reported confidence:", check["conf_matches"].all(),
      "| reproduces reported hard-label rate:", check["hard_matches"].all())
